# 07 — Work-Regime / Occupational-Mobility Archetypes

Research-only integration audit. This notebook does **not** infer job title, occupation, employment status, or true workplace.

It composes Stage 03a + 05b + optional 05c + 06 evidence into descriptive WORK representation regimes.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = 'https://github.com/tanh1c/geolife.git'
BRANCH = 'eda/07-work-regime-archetypes'
REPO_DIR = Path('/root/geolife')

if not (REPO_DIR / '.git').exists():
    subprocess.run(['git','clone','--branch',BRANCH,'--single-branch',REPO_URL,str(REPO_DIR)], check=True)
else:
    subprocess.run(['git','-C',str(REPO_DIR),'fetch','origin',BRANCH], check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'checkout',BRANCH], check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'reset','--hard',f'origin/{BRANCH}'], check=True)

print('repo:', REPO_DIR)
print('sha:', subprocess.run(['git','-C',str(REPO_DIR),'rev-parse','--short','HEAD'], check=True, capture_output=True, text=True).stdout.strip())


In [ ]:
from importlib.util import module_from_spec, spec_from_file_location
import pandas as pd

def load_module(name, path):
    spec = spec_from_file_location(name, path)
    assert spec is not None and spec.loader is not None
    module = module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

stage07 = load_module('stage07', REPO_DIR / 'analysis' / '07_work_regime_archetypes.py')
print(stage07.synthetic_self_check())


In [ ]:
VOLUME_ROOT = Path('/mnt/geolife-data')
CACHE_07 = VOLUME_ROOT / 'cache' / '07_work_regime_archetypes'
CACHE_07.mkdir(parents=True, exist_ok=True)

def find_required(filename, preferred=()):
    for path in preferred:
        if path.exists():
            return path
    matches = sorted(VOLUME_ROOT.glob(f'**/{filename}'))
    if matches:
        return matches[0]
    raise FileNotFoundError(f'Cannot find required artifact: {filename}')

def find_optional(filename, preferred=()):
    for path in preferred:
        if path.exists():
            return path
    matches = sorted(VOLUME_ROOT.glob(f'**/{filename}'))
    return matches[0] if matches else None

FEATURES_PATH = find_required('user_behavior_features.csv', [
    VOLUME_ROOT / 'cache' / '03a_user_behavior_deep_dive' / 'user_behavior_features.csv',
])
HOME_PATH = find_required('home_consensus_private.pkl', [
    VOLUME_ROOT / 'cache' / '05b_home_consensus_adaptive_work' / 'home_consensus_private.pkl',
])
WORK_PATH = find_required('adaptive_work_patterns_42_private.pkl', [
    VOLUME_ROOT / 'cache' / '05b_home_consensus_adaptive_work' / 'adaptive_work_patterns_42_private.pkl',
])
ROUTINE_PATH = find_required('user_summary_private.pkl', [
    VOLUME_ROOT / 'cache' / '06_routine_habit_mining' / 'user_summary_private.pkl',
])
INDEPENDENT_PATH = find_optional('candidate_comparison_private.pkl', [
    VOLUME_ROOT / 'cache' / '05c_stable_secondary_evidence' / 'candidate_comparison_private.pkl',
])

print('03a:', FEATURES_PATH)
print('05b home:', HOME_PATH)
print('05b work:', WORK_PATH)
print('06:', ROUTINE_PATH)
print('05c optional:', INDEPENDENT_PATH)


In [ ]:
behavior_features = pd.read_csv(FEATURES_PATH, dtype={'user_id': str})
home_evidence = pd.read_pickle(HOME_PATH)
work_patterns = pd.read_pickle(WORK_PATH)
routine_users = pd.read_pickle(ROUTINE_PATH)
independent = pd.read_pickle(INDEPENDENT_PATH) if INDEPENDENT_PATH else None

# Some older 03a caches may predate candidate enrichment. Reuse the exact 03a helper if needed.
needed_flags = {'stable_shifted_candidate', 'mobile_work_like_candidate'}
if not needed_flags.issubset(behavior_features.columns):
    try:
        import geolife  # noqa: F401
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], cwd=REPO_DIR, check=True)
    stage03a = load_module('stage03a_for_07', REPO_DIR / 'analysis' / '03a_user_behavior_deep_dive.py')
    behavior_features = stage03a.assign_behavioral_candidates(behavior_features)

print('behavior users:', behavior_features['user_id'].astype(str).nunique())
print('work-pattern users:', work_patterns['user_id'].astype(str).nunique())
print('routine users:', routine_users['user_id'].astype(str).nunique())
print('independent-evidence users:', 0 if independent is None else independent['user_id'].astype(str).nunique())


In [ ]:
audit = stage07.run_audit(
    behavior_features,
    work_patterns,
    routine_users,
    home_evidence=home_evidence,
    independent_secondary_evidence=independent,
)

print('ARCHETYPE SUMMARY')
display(audit.archetype_summary)

print('WORK REPRESENTATION SUMMARY')
display(audit.representation_summary)

print('UPSTREAM OVERLAP')
display(audit.upstream_overlap)

print('EVIDENCE SUMMARY')
display(audit.evidence_summary)


In [ ]:
# User-level tables stay private on the Modal Volume.
audit.evidence_matrix.to_pickle(CACHE_07 / 'evidence_matrix_private.pkl')
audit.archetypes.to_pickle(CACHE_07 / 'work_regime_archetypes_private.pkl')

for name, frame in {
    'archetype_summary': audit.archetype_summary,
    'representation_summary': audit.representation_summary,
    'upstream_overlap': audit.upstream_overlap,
    'evidence_summary': audit.evidence_summary,
}.items():
    frame.to_csv(CACHE_07 / f'{name}.csv', index=False)

print('saved:', CACHE_07)


## Interpretation boundary

- `fixed_site_like` is not OFFICE ground truth.
- `route_centric_mobile_like` is not a driver/sales/courier occupation label.
- `multi_site_recurring` is not evidence of multiple offices.
- If the archetypes are coherent, the next independent axis should be coarse external POI / land-use context.
